# 01 Dataset check

Verify the local FreiHAND dataset, inspect the available RGB variants, confirm the canonical train/validation split, and check TensorFlow batch loading.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

cwd = Path.cwd().resolve()
if (cwd / 'src').exists():
    project_root = cwd
elif (cwd.parent / 'src').exists():
    project_root = cwd.parent
else:
    raise RuntimeError('Run this notebook from the repository root or notebooks directory.')

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'Project root: {project_root}')


In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf

from src.data.freihand import (
    FreiHand,
    SPLIT_SEED,
    SPLIT_VALIDATION_FRACTION,
    VARIANTS,
)


## Validate dataset paths


In [ ]:
dataset = FreiHand()
dataset.validate()

eval_dataset = FreiHand(split='eval')
eval_dataset.validate()

print(f'Training root: {dataset.root}')
print(f'Training samples: {dataset.sample_count:,}')
print(f'Evaluation samples: {eval_dataset.sample_count:,}')
print(f'Variants: {VARIANTS}')


## Inspect FreiHAND image variants


In [ ]:
sample_id = 0
for variant in VARIANTS:
    sample = dataset.sample(sample_id, variant=variant, load_image=False)
    print(f'{variant:>6}: image_id={sample.image_id:06d}, file={sample.image_path.name}')

fig, axes = plt.subplots(1, len(VARIANTS), figsize=(3.2 * len(VARIANTS), 3.2))
for ax, variant in zip(axes, VARIANTS):
    dataset.plot_sample(sample_id, variant=variant, image_size=(224, 224), ax=ax)
fig.tight_layout()
plt.show()


## Check the canonical split


In [ ]:
train_idx, val_idx = dataset.train_validation_split(
    validation_fraction=SPLIT_VALIDATION_FRACTION,
    seed=SPLIT_SEED,
)

print(f'Train samples: {len(train_idx):,}')
print(f'Validation samples: {len(val_idx):,}')
print(f'First validation sample ids: {val_idx[:8].tolist()}')

assert len(train_idx) + len(val_idx) == dataset.sample_count
assert len(set(train_idx).intersection(set(val_idx))) == 0


## Check TensorFlow loading


In [ ]:
tf_ds = dataset.tf_dataset(
    indices=train_idx[:64],
    variants='gs',
    batch_size=16,
    image_size=(224, 224),
    shuffle=True,
    seed=SPLIT_SEED,
    flatten_keypoints=True,
)

images, keypoints = next(iter(tf_ds))
print(f'Images: {images.shape}, {images.dtype}, min={float(tf.reduce_min(images)):.3f}, max={float(tf.reduce_max(images)):.3f}')
print(f'Keypoints: {keypoints.shape}, {keypoints.dtype}')

assert tuple(images.shape) == (16, 224, 224, 3)
assert tuple(keypoints.shape) == (16, 42)
